### Setup


In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [2]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az

from prettytable import PrettyTable

from src.stat_utils import *
from src.anl_utils import load_data, get_session_data

### Load and prepare data

In [3]:
sim_results_folder = '../results/simulation'
data_folder = '../data'
sync_at_file = os.path.join(sim_results_folder, 'first_session_arnold_tongues.npy')
emp_at_file = os.path.join(data_folder, 'Experiment.csv')

In [4]:
def fit_transform(df, columns, sessions):
    """Z-score transform specified columns within specified sessions.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    columns (list): List of column names to be transformed.
    sessions (list): List of session identifiers (int) to consider for mean and std calculation.
    
    """
    transformed_df = df.copy()
    session_mask = df['SessionID'].isin(sessions)
    for column in columns:
        mean = df.loc[session_mask, column].mean()
        std = df.loc[session_mask, column].std()
        transformed_df[column] = (transformed_df[column] - mean) / std
    return transformed_df



In [ ]:
# Load the simulations results and the empirical data
sync_results = np.load(sync_at_file)
sync_results_vector = sync_results.mean(axis=0).flatten()
data = load_data(emp_at_file)

# Dummy code session 9
data_transfer = data.copy()
data_transfer['Transfer'] = (data_transfer['SessionID'] == 9)

# Filter data for learning phase (Sessions 1 to 8)
data_learning = data[data['SessionID'] <= 8].copy()

# Map synchrony values to each condition in DataFrame
data_learning['Synchrony'] = data_learning['Condition'].apply(lambda x: sync_results_vector[x-1])

# Z-score the relevant columns
data_learning = zscore_data(data_learning, ['ContrastHeterogeneity', 'GridCoarseness', 'Synchrony'])
data_transfer = fit_transform(data_transfer, ['ContrastHeterogeneity', 'GridCoarseness'], sessions=[1,2,3,4,5,6,7,8])

# Center the session number
mean_session = data_learning['SessionID'].mean()
data_learning['SessionCentered'] = data_learning['SessionID'] - mean_session
data_transfer['SessionCentered'] = data_transfer['SessionID'] - mean_session

### Define statistical models

In [6]:
model_features = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + SessionCentered * (ContrastHeterogeneity + GridCoarseness) +  (1 + SessionCentered + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data_learning,
    family="bernoulli"
)

Does learning occur (i.e., does session have an effect on performance)? 

Do the effects of contrast heterogeneity and/or grid coarseness depend on session?

In [7]:
idata_features = model_features.fit(
    draws=2000, tune=2000, target_accept=0.9,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
/home/mario/miniconda3/envs/bat_env/lib/python3.13/site-packages/pytensor/link/c/cmodule.py:2968: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, SessionCentered, SessionCentered:ContrastHeterogeneity, Ses

In [16]:
predictors = ["SessionCentered", "ContrastHeterogeneity", "GridCoarseness", "SessionCentered:ContrastHeterogeneity","SessionCentered:GridCoarseness", "ContrastHeterogeneity:GridCoarseness"]
directions = ['greater', 'less', 'less','less','less', 'greater']

posterior = posterior_table(idata_features, predictors, directions)

odds_ratios = OR_table(idata_features, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_features, var_names=predictors, hdi_prob=0.95)

One-sided posterior probabilities:
+---------------------------------------+-----------+-------+
|               Predictor               | direction |   P   |
+---------------------------------------+-----------+-------+
|            SessionCentered            |  greater  | 1.000 |
|         ContrastHeterogeneity         |    less   | 1.000 |
|             GridCoarseness            |    less   | 1.000 |
| SessionCentered:ContrastHeterogeneity |    less   | 1.000 |
|     SessionCentered:GridCoarseness    |    less   | 0.955 |
|  ContrastHeterogeneity:GridCoarseness |  greater  | 1.000 |
+---------------------------------------+-----------+-------+

Odds ratios:
+---------------------------------------+-------+--------------+---------------+
|               Predictor               |  Mean | Lower (2.5%) | Upper (97.5%) |
+---------------------------------------+-------+--------------+---------------+
|            SessionCentered            | 1.099 |    1.068     |     1.132     |
|      

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
SessionCentered,0.095,0.014,0.066,0.123,0.000,0.000,4153.0,4739.0,1.0
ContrastHeterogeneity,-0.949,0.131,-1.223,-0.696,0.003,0.002,2266.0,2774.0,1.0
GridCoarseness,-0.317,0.029,-0.374,-0.260,0.000,0.000,3828.0,4431.0,1.0
SessionCentered:ContrastHeterogeneity,-0.081,0.005,-0.091,-0.070,0.000,0.000,10696.0,5888.0,1.0
SessionCentered:GridCoarseness,-0.008,0.005,-0.018,0.001,0.000,0.000,9752.0,5916.0,1.0
ContrastHeterogeneity:GridCoarseness,0.272,0.034,0.202,0.341,0.001,0.001,3650.0,3918.0,1.0


In [9]:
def session_simple_effects_CH(idata, sessions, hdi=0.95):
    # Extract posterior draws for CH main and CH:Session interaction
    posterior = az.extract(idata, var_names=["ContrastHeterogeneity",
                                        "SessionCentered:ContrastHeterogeneity"]).to_dataframe()
    beta_contrast_heterogeneity   = posterior["ContrastHeterogeneity"].to_numpy()
    beta_session_ch_interaction = posterior["SessionCentered:ContrastHeterogeneity"].to_numpy()

    rows = []
    for session in sessions:
        beta_draws = beta_contrast_heterogeneity + beta_session_ch_interaction  * session
        odds_ratio_draws   = np.exp(beta_draws)

        hdi_low, hdi_high = az.hdi(beta_draws, hdi_prob=hdi)
        odds_ratio_low, odds_ratio_high = az.hdi(odds_ratio_draws, hdi_prob=hdi)

        rows.append({
            "Session": session,
            "beta_mean": float(beta_draws.mean()),
            f"beta_hdi_{int((1-hdi)/2*100)}%": float(hdi_low),
            f"beta_hdi_{int((1+hdi)/2*100)}%": float(hdi_high),
            "OR_mean": float(odds_ratio_draws.mean()),
            f"OR_hdi_{int((1-hdi)/2*100)}%": float(odds_ratio_low),
            f"OR_hdi_{int((1+hdi)/2*100)}%": float(odds_ratio_high),
            "Pr_beta_less_0": float((beta_draws < 0).mean())
        })
    return pd.DataFrame(rows)

    
sessions = np.arange(1, 9) - 4.5
tbl_ch_by_session = session_simple_effects_CH(idata_features, sessions, hdi=0.95)
print(tbl_ch_by_session)


   Session  beta_mean  beta_hdi_2%  beta_hdi_97%   OR_mean  OR_hdi_2%  \
0     -3.5  -0.666813    -0.928989     -0.398095  0.517877   0.378583   
1     -2.5  -0.747505    -1.009615     -0.479621  0.477694   0.347816   
2     -1.5  -0.828197    -1.098239     -0.571476  0.440642   0.330479   
3     -0.5  -0.908889    -1.177184     -0.650235  0.406477   0.305656   
4      0.5  -0.989581    -1.249010     -0.721815  0.374971   0.275707   
5      1.5  -1.070273    -1.323104     -0.795779  0.345917   0.254308   
6      2.5  -1.150965    -1.412863     -0.885632  0.319124   0.237663   
7      3.5  -1.231657    -1.488673     -0.961445  0.294415   0.218903   

   OR_hdi_97%  Pr_beta_less_0  
0    0.650326        0.999625  
1    0.598849        0.999875  
2    0.561067        1.000000  
3    0.517900        1.000000  
4    0.471534        1.000000  
5    0.434864        1.000000  
6    0.404013        1.000000  
7    0.372668        1.000000  


### Transfer session analysis

In [11]:
model_transfer = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + Transfer + SessionCentered * (ContrastHeterogeneity + GridCoarseness) +  (1 + SessionCentered + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data_transfer,
    family="bernoulli"
)

In [12]:
idata_transfer = model_transfer.fit(
    draws=2000, tune=2000, target_accept=0.9,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, Transfer, SessionCentered, SessionCentered:ContrastHeterogeneity, SessionCentered:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, SessionCentered|SubjectID_sigma, SessionCentered|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 3933 seconds.
There were 6 divergences after tuning. Increase `target_accept` or reparameterize.


In [13]:
predictors = ["SessionCentered", "ContrastHeterogeneity", "GridCoarseness", "SessionCentered:ContrastHeterogeneity","SessionCentered:GridCoarseness", "ContrastHeterogeneity:GridCoarseness", "Transfer"]

az.summary(idata_transfer, var_names=predictors, hdi_prob=0.95)

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
SessionCentered,0.079,0.008,0.063,0.096,0.000,0.000,5288.0,4802.0,1.0
ContrastHeterogeneity,-0.923,0.137,-1.193,-0.647,0.003,0.003,2460.0,3070.0,1.0
GridCoarseness,-0.305,0.032,-0.367,-0.240,0.001,0.001,3700.0,4344.0,1.0
SessionCentered:ContrastHeterogeneity,-0.047,0.004,-0.055,-0.038,0.000,0.000,9656.0,5853.0,1.0
SessionCentered:GridCoarseness,-0.001,0.004,-0.009,0.007,0.000,0.000,10094.0,6187.0,1.0
ContrastHeterogeneity:GridCoarseness,0.266,0.033,0.199,0.330,0.001,0.001,3639.0,4090.0,1.0
Transfer,-0.429,0.040,-0.512,-0.355,0.000,0.000,10548.0,5777.0,1.0


In [14]:
def post_mean_prob_pop(model, idata, test_data):
    """
    Population-level posterior of mean accuracy for the given test_data.
    Uses kind="mean" (expected probability) and excludes group-specific effects.
    Returns: array of shape (n_draws,) with the mean probability per draw.
    """
    pp = model.predict(
        idata=idata,
        data=test_data,
        kind="mean",
        include_group_specific=False,  # <- population level: no REs
        inplace=False
    )
    # Bambi stores this in the posterior group with a var like "<response>_mean".
    # Grab the first *_mean variable defensively.
    mean_vars = [v for v in pp.posterior.data_vars if v.endswith("_mean")]
    if not mean_vars:
        # fallback: some versions use "y_mean"
        mean_vars = [v for v in pp.posterior.data_vars if v == "y_mean"]
    varname = mean_vars[0]

    arr = pp.posterior[varname].values  # shape: (chains, draws, obs)
    draws = arr.shape[0] * arr.shape[1]
    arr2 = arr.reshape(draws, arr.shape[2])  # (draws, obs)
    return arr2.mean(axis=1)  # average across obs -> one mean prob per draw

In [15]:
# Posterior distributions of mean accuracy (population level)

data_s9 = get_session_data(data_transfer, 9)
data_s2 = get_session_data(data_transfer, 2)

posterior_s9 = post_mean_prob_pop(model_transfer, idata_transfer, data_s9)
posterior_s2 = post_mean_prob_pop(model_transfer, idata_transfer, data_s2)

# Contrasts & summaries
def summarize_diff(a, b, hdi=0.95, rope=0.02):
    d = a - b
    h = az.hdi(d, hdi_prob=hdi)
    return {
        "P(a<b)": float((a < b).mean()),
        "mean_diff": float(d.mean()),
        "hdi_low": float(h[0]),
        "hdi_high": float(h[1]),
        "P_within_ROPE(±{:.0%})".format(rope): float((np.abs(d) < rope).mean())
    }

print("S9 vs S2:", summarize_diff(posterior_s9, posterior_s2))

S9 vs S2: {'P(a<b)': 0.3795, 'mean_diff': 0.0030842228091367135, 'hdi_low': -0.018058326908948708, 'hdi_high': 0.02437592190652349, 'P_within_ROPE(±2%)': 0.92875}
